# ResNet18 Cloud Filter — NASA GLOBE Ac Folder

Uses the trained multi-head ResNet18 (head2) to identify which images in the NASA GLOBE `Ac` folder actually contain clouds. Many images are non-cloud because the collection process captured all cardinal directions, not just skyward.

In [10]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torchvision.models import ResNet18_Weights
from pathlib import Path
import numpy as np

class MultiHeadResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.head1 = base.fc
        self.head2 = nn.Linear(512, num_classes)
        for param in self.backbone.parameters():
            param.requires_grad = False
        for param in self.head1.parameters():
            param.requires_grad = False

    def forward(self, x):
        features = self.backbone(x).flatten(1)
        return self.head1(features), self.head2(features)

SAVE_PATH = Path("../models/multihead_resnet18_cloud_sky.pth")
checkpoint = torch.load(SAVE_PATH, map_location="cpu", weights_only=False)
classes = checkpoint["classes"]   # ['clear_sky', 'cloud']
cloud_idx = classes.index("cloud")

imagenet_classes = ResNet18_Weights.IMAGENET1K_V1.meta["categories"]

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = MultiHeadResNet18(num_classes=len(classes))
model.load_state_dict(checkpoint["model_state_dict"])
model.eval().to(device)

print(f"Model loaded. Classes: {classes}, device: {device}")
print(f"ImageNet classes loaded: {len(imagenet_classes)}")

Model loaded. Classes: ['clear_sky', 'cloud'], device: mps
ImageNet classes loaded: 1000


In [ ]:
import cv2
from tqdm.auto import tqdm
from torch.nn.functional import softmax

AC_PATH = Path("../resources/cloud-images/NASA_GLOBE_CD/downloaded_images/Ac")
CONF_THRESHOLD = 0.8
BATCH_SIZE = 64

preprocess = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

image_paths = sorted(AC_PATH.glob("*.jpg")) + sorted(AC_PATH.glob("*.png"))
print(f"Found {len(image_paths)} images in Ac folder")

filenames, head2_preds, head2_confs = [], [], []
head1_preds, head1_confs = [], []

with torch.no_grad():
    for start in tqdm(range(0, len(image_paths), BATCH_SIZE)):
        batch_paths = image_paths[start:start + BATCH_SIZE]
        tensors = []
        valid_paths = []
        for p in batch_paths:
            img = cv2.imread(str(p))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            tensors.append(preprocess(img))
            valid_paths.append(p)

        if not tensors:
            continue

        batch = torch.stack(tensors).to(device)
        out1, out2 = model(batch)
        probs2 = softmax(out2, dim=1).cpu().numpy()
        probs1 = softmax(out1, dim=1).cpu().numpy()

        for p, prob2, prob1 in zip(valid_paths, probs2, probs1):
            top1_idx = int(np.argmax(prob1))
            filenames.append(p.name)
            head2_preds.append(classes[int(np.argmax(prob2))])
            head2_confs.append(float(prob2[cloud_idx]))
            head1_preds.append(imagenet_classes[top1_idx])
            head1_confs.append(float(prob1[top1_idx]))

print(f"\nProcessed {len(filenames)} images")

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "filename":   filenames,
    "head2_pred": head2_preds,
    "head2_conf": head2_confs,
    "head1_pred": head1_preds,
    "head1_conf": head1_confs,
})

n_cloud    = (results_df["head2_pred"] == "cloud").sum()
n_noncloud = (results_df["head2_pred"] == "clear_sky").sum()
n_keep     = (results_df["head2_conf"] >= CONF_THRESHOLD).sum()

print(f"Total images:          {len(results_df):,}")
print(f"Predicted cloud:       {n_cloud:,} ({100*n_cloud/len(results_df):.1f}%)")
print(f"Predicted not cloud:   {n_noncloud:,} ({100*n_noncloud/len(results_df):.1f}%)")
print(f"Keep (conf ≥ {CONF_THRESHOLD}):     {n_keep:,} ({100*n_keep/len(results_df):.1f}%)")

## Visualisation 1: Confidence Distribution

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(results_df["head2_conf"], bins=50, color="steelblue", edgecolor="white")
ax.axvline(CONF_THRESHOLD, color="red", linestyle="--", label=f"threshold = {CONF_THRESHOLD}")
ax.set_xlabel("head2 cloud confidence score")
ax.set_ylabel("Number of images")
ax.set_title("Distribution of cloud confidence scores — NASA GLOBE Ac folder")
ax.legend()
plt.tight_layout()
plt.show()

## Visualisation 2: Sample Images Predicted as NOT Cloud

In [ ]:
non_cloud_df = results_df[results_df["head2_pred"] == "clear_sky"].sort_values("head2_conf")
sample = non_cloud_df.head(16)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img = cv2.cvtColor(cv2.imread(str(AC_PATH / row["filename"])), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"head2_conf: {row['head2_conf']:.2f}\n{row['filename'][:30]}", fontsize=6)

plt.suptitle("Sample images predicted as NOT cloud (lowest confidence first)", fontsize=12)
plt.tight_layout()
plt.show()

## Visualisation 3: Borderline Images (cloud confidence 0.5 – 0.8)

In [ ]:
borderline_df = results_df[
    (results_df["head2_conf"] >= 0.5) & (results_df["head2_conf"] < CONF_THRESHOLD)
]
sample = borderline_df.sample(min(16, len(borderline_df)), random_state=42)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img = cv2.cvtColor(cv2.imread(str(AC_PATH / row["filename"])), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(
        f"head2_pred: {row['head2_pred']} ({row['head2_conf']:.2f})\n"
        f"head1_pred: {row['head1_pred']} ({row['head1_conf']:.2f})\n"
        f"{row['filename'][:28]}",
        fontsize=6,
    )

for ax in list(axes.flat)[len(sample):]:
    ax.axis("off")

plt.suptitle("Borderline images (head2_conf 0.5 – 0.8)", fontsize=12)
plt.tight_layout()
plt.show()

## Visualisation 3: Sample Images Predicted as Cloud

In [ ]:
cloud_df = results_df[results_df["head2_conf"] >= CONF_THRESHOLD].sort_values("head2_conf", ascending=False)
sample = cloud_df.head(16)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img = cv2.cvtColor(cv2.imread(str(AC_PATH / row["filename"])), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"head2_conf: {row['head2_conf']:.2f}\n{row['filename'][:30]}", fontsize=6)

plt.suptitle(f"Sample images predicted as cloud (head2_conf ≥ {CONF_THRESHOLD})", fontsize=12)
plt.tight_layout()
plt.show()

## Export: Clean Image Filenames

Run this cell after reviewing the visualisations above. Exports filenames of images the model is confident are cloud images.

In [ ]:
EXPORT_PATH = Path("../resources/cloud-images/NASA_GLOBE_CD/ac_clean_filenames.txt")

clean = results_df[results_df["head2_conf"] >= CONF_THRESHOLD]["filename"].tolist()

with open(EXPORT_PATH, "w") as f:
    for name in clean:
        f.write(name + "\n")

print(f"Exported {len(clean):,} clean filenames to {EXPORT_PATH}")